In [37]:
import re
from pathlib import Path
import csv
from collections import Counter
from difflib import SequenceMatcher

# === Mode Selection ====
mode = "drama"  # change to "novel" or "drama_sentence" as needed

# === Drama Mode Restrictions ===
'''These are words that aren't necessarily speaker labels, etc'''
EXCLUDE_WORDS = {"INDIMA", "IMIBUZO", "KWENGXOXO", "ISINXUMEZELELO", 
                 "ITS", "ID", "PE", "EQ", "OV", "OFUNA", "EMIFUTSHANE",
                 "IMPENDULO", "PHENDULA", "NEEMFUNO"}  # add more if needed

# === File paths ===
TEXT_FILE = "Kubanjenwe_Ngazo_Enxuba.txt"           # Name of the output file
file_dir = Path.cwd() / "raw_outputs" / TEXT_FILE   # Name of the output directory
out_dir = Path.cwd() / "final_corpus"               # Name of the final-corpus directory
out_dir.mkdir(parents=True, exist_ok=True)

In [32]:
def common_cleaning(text: str) -> str:
    """Apply base cleaning rules for OCR text."""
    # 1. Remove non-Latin characters but keep punctuation
    text = re.sub(r"[^a-zA-Z0-9\s\.?\!\"',;:\-\(\)]", "", text)
    # 2. Remove isolated page numbers
    text = re.sub(r"^\d+\s*$", "", text, flags=re.MULTILINE)
    # 3. Handle hyphenation across lines
    text = re.sub(r"-\s*\n\s*", "", text)
    # 4. Normalize multiple line breaks
    text = re.sub(r"\n{2,}", "\n", text)
    # 5. Remove spaces before punctuation
    text = re.sub(r"\s+([\.!?;:,])", r"\1", text)
    return text

In [33]:
def discover_and_normalize_characters(text: str, similarity_threshold: float = 0.8, log_dir: Path = None):
    """
    Discover likely speaker names by filtering uppercase tokens.
    """
    name_counts = Counter()
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        # Match uppercase tokens at start, possibly with colon
        match = re.match(r"^([A-Z]+)\s*:?(\s*)", line)
        if match:
            token = match.group(1)
            token = re.sub(r"[^\w]", "", token)  # strip punctuation

            # === NEW FILTERS ===
            if token in EXCLUDE_WORDS:
                continue
            if len(token) < 3:
                continue  # ignore single/double letters unless we whitelist later
            if token.startswith("U") and len(token) > 3:
                token = token[1:]  # strip leading 'U' (e.g. UZENZILE -> ZENZILE)

            name_counts[token] += 1
        else:
            # Split line into words and take first uppercase word if valid
            words = line.split()
            if words and words[0].isupper():
                token = re.sub(r"[^\w]", "", words[0])
                if token in EXCLUDE_WORDS:
                    continue
                if len(token) < 3:
                    continue
                if token.startswith("U") and len(token) > 3:
                    token = token[1:]
                name_counts[token] += 1

    # --- Group similar names by frequency ---
    canonical_names = []
    name_map = {}
    for name, _ in name_counts.most_common():
        matched = False
        for canon in canonical_names:
            if SequenceMatcher(None, name, canon).ratio() >= similarity_threshold:
                name_map[name] = canon
                matched = True
                break
        if not matched:
            canonical_names.append(name)
            name_map[name] = name

    # Logging remains the same
    if log_dir:
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / "character_normalization_log.csv"
        with open(log_path, "w", newline="", encoding="utf-8") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["Variant", "Count", "Canonical"])
            for variant, count in name_counts.most_common():
                writer.writerow([variant, count, name_map.get(variant, variant)])
        print(f"[LOG] Character normalization log saved to: {log_path}")

    return name_map, sorted(canonical_names, key=lambda x: name_counts[x], reverse=True)

In [34]:
def process_drama(text: str):
    """
    Process text in drama mode:
    - Detects and normalizes speaker names.
    - Groups their dialogue until an empty line or next speaker.
    - Splits narrative (non-speaker) blocks into one sentence per line.
    """
    name_map, canonical_names = discover_and_normalize_characters(text)

    output_lines = []
    buffer = []
    current_speaker = None
    current_dialogue = []

    def flush_narrative(buf):
        if not buf:
            return []
        joined = " ".join(buf)
        sentences = re.split(r'(?<=[.!?])\s+', joined)
        return [s.strip() for s in sentences if s.strip()]

    for line in text.splitlines():
        raw_line = line
        line = line.strip()

        # --- empty line: close speaker dialogue ---
        if not line:
            if current_speaker and current_dialogue:
                output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
                current_speaker = None
                current_dialogue = []
            continue

        # --- speaker line ---
        match = re.match(r"^([A-Z]+)\s*:?(.*)", line)
        if match:
            raw_name = re.sub(r"[^\w]", "", match.group(1))
            if raw_name in name_map:
                # flush any buffered narrative before new speaker
                if buffer:
                    output_lines.extend(flush_narrative(buffer))
                    buffer = []

                # flush old speaker
                if current_speaker and current_dialogue:
                    output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
                    current_dialogue = []

                # start new speaker
                current_speaker = name_map[raw_name]
                remainder = match.group(2).strip()
                if remainder:
                    current_dialogue.append(remainder)
                continue

        # --- non-speaker text ---
        if current_speaker:
            current_dialogue.append(line)
        else:
            buffer.append(line)

    # --- flush leftovers ---
    if current_speaker and current_dialogue:
        output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
    if buffer:
        output_lines.extend(flush_narrative(buffer))

    return "\n".join(output_lines), canonical_names

In [ ]:
def process_drama_sentence_mode(text: str) -> str:
    """
    Processes drama-like text by:
    - Removing speaker labels entirely
    - Splitting everything into sentences
    - Returning clean sentence-by-sentence text
    """
    # 1. Remove speaker labels (uppercase names, optional numbers, colon, etc.)
    text_no_labels = re.sub(r'^[A-ZÀ-Ý][A-ZÀ-Ý0-9\s\-]*\s?:\s?', '', text, flags=re.MULTILINE)

    # 2. Flatten text but keep paragraph breaks (double newlines)
    text_flat = re.sub(r'\n{2,}', '\n\n', text_no_labels)  # keep double newlines
    text_flat = re.sub(r'\n', ' ', text_flat)  # replace single newlines with spaces

    # 3. Split into sentences (keep punctuation)
    sentences = re.split(r'(?<=[.!?])\s+', text_flat)

    # 4. Clean up each sentence
    sentences = [s.strip() for s in sentences if s.strip()]

    # 5. Return each sentence on its own line
    return '\n'.join(sentences)

In [35]:
def process_novel(text: str):
    """Process text in novel mode: one sentence per line."""
    text = re.sub(r"\n+", " ", text)
    sentences = re.split(r'([\.!?]["\']?)', text)

    cleaned_sentences = []
    current = ""
    for part in sentences:
        current += part.strip() + " "
        if re.fullmatch(r'[\.!?]["\']?', part.strip()):
            cleaned_sentences.append(current.strip())
            current = ""
    if current.strip():
        cleaned_sentences.append(current.strip())

    return "\n".join(cleaned_sentences)

In [36]:
# === Directories ===
log_dir = Path.cwd() / "logs"

# === Load raw text ===
with open(file_dir, "r", encoding="utf-8") as f:
    raw_text = f.read()

# === Preprocess ===
cleaned_text = common_cleaning(raw_text)

if mode == "drama":
    # Pass log_dir so a CSV is generated
    processed_text, characters = process_drama(cleaned_text)
    print("Characters found:", characters)
elif mode == "drama_sentence":
    # New mode: remove speaker labels, split everything into sentences
    processed_text = process_drama_sentence_mode(cleaned_text)
    characters = None  # no character list in this mode
else:  # fallback to novel mode
    processed_text = process_novel(cleaned_text)
    characters = None

# === Save output ===
out_path = out_dir / f"{file_dir.stem}_cleaned.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(processed_text)

print(f"Pre-filter processing complete: {out_path}")

Characters found: ['LUNGILE', 'XHESHILE', 'FIKILE', 'FUNEKA', 'FEZILE', 'THEMBAKAZI', 'SAKHUMZI', 'MZIMKHULU', 'SABELO', 'SANDI', 'ZENZILE', 'MANDLANGISA', 'LAWUKAZI', 'ENGENTLA', 'COSAS', 'MFO', 'MBUZO', 'YUU', 'SITHATHWE', 'NGAZO', 'UPF', 'LUNGHE', 'BUTHUMA']
Pre-filter processing complete: /home/hlatsiieyhax/DevBlock/godhand_development/inhouse-tools/finalcorpusnator/final_corpus/Kubanjenwe_Ngazo_Enxuba_cleaned.txt
